<a href="https://colab.research.google.com/github/ayushunhale10-code/AI-ML-Assignments/blob/main/Assignment-09/Assignment_09_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 9: Object Detection from CCTV Footage

## Aim
To perform object detection on fixed-camera CCTV footage using a pretrained YOLO model.

## Theory

Object detection is a computer vision technique used to identify and locate objects in images and videos. YOLO (You Only Look Once) is a pretrained model that detects objects and marks them with bounding boxes and class labels.

In this practical, a fixed-camera CCTV video is converted from WebM to MP4 format and processed using YOLO.

In [6]:
!pip install -q ultralytics
!apt-get -qq update && apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.8 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [7]:
import os
import subprocess
import glob
from google.colab import files
from IPython.display import Video, display
from ultralytics import YOLO

print("Libraries ready.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Libraries ready.


In [8]:
uploaded = files.upload()
video_path = next(iter(uploaded.keys()))

print("Uploaded:", video_path)

Saving 2026_White_House_Correspondents'_Dinner_shooting_CCTV_Footage.webm to 2026_White_House_Correspondents'_Dinner_shooting_CCTV_Footage.webm
Uploaded: 2026_White_House_Correspondents'_Dinner_shooting_CCTV_Footage.webm


In [9]:
mp4_path = "/content/cctv_input.mp4"

result = subprocess.run([
    "ffmpeg", "-y",
    "-i", video_path,
    "-vf", "scale=-2:480",
    "-c:v", "libx264",
    "-preset", "ultrafast",
    "-crf", "28",
    "-an",
    "-movflags", "+faststart",
    mp4_path
], capture_output=True, text=True)

if result.returncode == 0:
    print("Conversion successful!")
    print("MP4 size:", round(os.path.getsize(mp4_path) / (1024 * 1024), 2), "MB")
else:
    print(result.stderr[-2000:])

Conversion successful!
MP4 size: 1.88 MB


In [10]:
display(Video(mp4_path, embed=True, width=500))

In [11]:
model = YOLO("yolo11n.pt")
print("YOLO model loaded.")

YOLO model loaded.


In [12]:
output_dir = "/content/detection_output"

model.predict(
    source=mp4_path,
    save=True,
    conf=0.25,
    imgsz=640,
    project=output_dir,
    name="cctv",
    exist_ok=True
)

print("Object detection completed.")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/251) /content/cctv_input.mp4: 512x640 7 persons, 1 suitcase, 469.9ms
video 1/1 (frame 2/251) /content/cctv_input.mp4: 512x640 8 persons, 1 suitcase, 180.0ms
video 1/1 (frame 3/251) /content/cctv_input.mp4: 512x640 8 persons, 1 suitcase, 154.0ms
video 1/1 (frame 4/251) /content/cctv_input.mp4: 512x640 7 persons, 1 suitcase, 146.4ms
video 1/1 (frame 5/251) /content/cctv_input.mp4: 512x640 7 persons, 1 suitcase, 153.0ms
video 1/1 (frame 6/

In [13]:
video_files = glob.glob(
    "/content/detection_output/cctv/*.avi"
) + glob.glob(
    "/content/detection_output/cctv/*.mp4"
)

print("Detected output files:", video_files)

if video_files:
    detected_video = video_files[0]
    print("Detected video path:", detected_video)
else:
    print("No output video found. Check Cell 9 for errors.")

Detected output files: ['/content/detection_output/cctv/cctv_input.avi']
Detected video path: /content/detection_output/cctv/cctv_input.avi


In [14]:
small_video = "/content/cctv_detected_final.mp4"

if video_files:
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", detected_video,
        "-vf", "scale=-2:480",
        "-c:v", "libx264",
        "-preset", "ultrafast",
        "-crf", "30",
        "-an",
        "-movflags", "+faststart",
        small_video
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print("Final MP4 created successfully.")
        print("Size:", round(os.path.getsize(small_video) / (1024 * 1024), 2), "MB")
    else:
        print(result.stderr[-2000:])
else:
    print("Run Cell 9 and check Cell 10 first.")

Final MP4 created successfully.
Size: 2.51 MB


In [15]:
if os.path.exists(small_video):
    display(Video(small_video, embed=True, width=500))
else:
    print("Final video not found.")

In [16]:
if os.path.exists(small_video):
    files.download(small_video)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Result Interpretation

The fixed-camera CCTV video was converted from WebM to MP4 format. The pretrained YOLO model detected recognizable objects and marked them with bounding boxes and class labels.

## Result

Object detection was performed on fixed-camera CCTV footage using a pretrained YOLO model. The processed video displayed detected objects with bounding boxes and labels.

## Conclusion

In this practical, YOLO was used to detect objects in fixed-camera CCTV footage. The input video was converted to MP4 format, processed, and saved with object labels and bounding boxes.